In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import glob
import json
import _utils

In [2]:
from statsmodels.distributions.empirical_distribution import ECDF

def ecdf_func(ensemble, obs):
    # np.nan is treated as Inf by ECDF, so need to manually remove these
    if np.isnan(ensemble).all():
        return(np.nan)
    if np.isnan(obs):
        return(np.nan)
    ensemble = ensemble[~np.isnan(ensemble)]
    
    return (ECDF(ensemble)(obs))

def ecdf_xr(ensemble, obs):       
    return xr.apply_ufunc(ecdf_func, ensemble, obs, 
                          input_core_dims = (["sim"], []), 
                          dask = "allowed", 
                          vectorize = True, ## required when function can only take 1D array
                         )

In [3]:
def test_ecdf(model_trends, obs_trend):
    ecdf_dat = []
    for simname in model_trends.sim.values:
        a = xr.concat([model_trends.drop_sel(sim = simname), obs_trend.expand_dims({"sim": ["obs"]})], dim = "sim")
        b = model_trends.sel(sim = simname)
        x = ecdf_xr(a, b)
        ecdf_dat.append([((b > 0) & (x == 0)).sum().values, ((b > 0) & (x == 1)).sum().values, (b > 0).sum().values, 
                         ((b < 0) & (x == 0)).sum().values, ((b < 0) & (x == 1)).sum().values, b.count().values])
    
    ecdf_dat = pd.DataFrame(ecdf_dat, columns = ["pos_ecdf_0", "pos_ecdf_1", "pos_trends", "neg_ecdf_0", "neg_ecdf_1", "num_values"])
    return(ecdf_dat)

In [4]:
def test_ecdf_subsample(model_trends, obs, N, seed = 123):
    np.random.seed(seed)
    ecdf_dat = []
    for simname in model_trends.sim.values:
        ## select N-member subset of ensemble
        ensemble_sub = model_trends.drop_sel(sim = simname)
        ind = np.random.randint(0, len(ensemble_sub.sim), size = N)
        a = ensemble_sub.isel(sim = ind)

        ## select current ensemble member
        b = model_trend.sel(sim = simname)
    
        x = ecdf_xr(a, b) ## ensemble member ecdf
        y = ecdf_xr(a, obs_trend) ## obs ecdf
        ecdf_dat.append([((b > 0) & (x == 0)).sum().values, ((b > 0) & (x == 1)).sum().values,
                         ((b < 0) & (x == 0)).sum().values, ((b < 0) & (x == 1)).sum().values,
                         (b > 0).sum().values, 
                        ((obs_trend > 0) & (y == 0)).sum().values, ((obs_trend > 0) & (y == 1)).sum().values, 
                        ((obs_trend < 0) & (y == 0)).sum().values, ((obs_trend < 0) & (y == 1)).sum().values])

    ecdf_dat = pd.DataFrame(ecdf_dat, columns = ["sim_pos_ecdf_0", "sim_pos_ecdf_1", "sim_neg_ecdf_0", "sim_neg_ecdf_1", "sim_pos_trends", 
                                                 "obs_pos_ecdf_0", "obs_pos_ecdf_1", "obs_neg_ecdf_0", "obs_neg_ecdf_1"])
    return(ecdf_dat)

### Base comparisons:

In [5]:
dir = "../mnt_processed_data/"

In [6]:
model_var_dict = json.load(open(dir+"model_var_dict.json"))

In [7]:
common_mask = xr.open_dataset("../processed_data/common_land_mask.nc")
def mask(ds):
    return(xr.where(common_mask.__xarray_dataarray_variable__ == 1, ds, np.nan))

In [8]:
## base time period, rx1day comparisons
start = 1979
end = 2020

for ensemble in ["cmip", "mesaclip", "spear"]:
    print(ensemble)
    model_trend = _utils.read_trends(dir, ensemble, "rx1day", start, end)
    
    for obs in ["cpc", "mswep"]:
        print(obs)
        obs_trend = _utils.read_trends(dir, obs, "rx1day", start, end)
        ecdf_result = ecdf_xr(model_trend, obs_trend)
        ecdf_result.to_netcdf(dir+"ecdf/"+obs+"_"+ensemble+"_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")

        ecdf_test = test_ecdf(mask(model_trend), mask(obs_trend))
        ecdf_test.to_csv(dir+"ecdf/"+obs+"_"+ensemble+"_rx1day_"+str(start)+"-"+str(end)+"_ecdf_test.csv")
        
        if ensemble == "cmip":
            ecdf_result = ecdf_xr(model_trend.sel(sim = model_var_dict["cmip_day_onevar"]), obs_trend)
            ecdf_result.to_netcdf(dir+"ecdf/"+obs+"_"+ensemble+"-sub_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")
            
            ecdf_test = test_ecdf(mask(model_trend.sel(sim = model_var_dict["cmip_day_onevar"])), mask(obs_trend))
            ecdf_test.to_csv(dir+"ecdf/"+obs+"_"+ensemble+"-sub_rx1day_"+str(start)+"-"+str(end)+"_ecdf_test.csv")
            

cmip
cpc
mswep
mesaclip
cpc
mswep
spear
cpc
mswep


In [9]:
## base time period, mon-p095 comparisons
start = 1979
end = 2020

for ensemble in ["cmip", "mesaclip", "spear"]:
    print(ensemble)
    model_trend = _utils.read_trends(dir, ensemble, "mon-p095", start, end)
    
    for obs in ["gpcc", "gpcp", "mswep"]:
        print(obs)
        obs_trend = _utils.read_trends(dir, obs, "mon-p095", start, end)
        ecdf_result = ecdf_xr(model_trend, obs_trend)
        ecdf_result.to_netcdf(dir+"ecdf/"+obs+"_"+ensemble+"_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

        ecdf_test = test_ecdf(mask(model_trend), mask(obs_trend))
        ecdf_test.to_csv(dir+"ecdf/"+obs+"_"+ensemble+"_mon-p095_"+str(start)+"-"+str(end)+"_ecdf_test.csv")
                    
        if ensemble == "cmip":
            ecdf_result = ecdf_xr(model_trend.sel(sim = model_var_dict["cmip_mon_onevar"]), obs_trend)
            ecdf_result.to_netcdf(dir+"ecdf/"+obs+"_"+ensemble+"-sub_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

            ecdf_test = test_ecdf(mask(model_trend.sel(sim = model_var_dict["cmip_mon_onevar"])), mask(obs_trend))
            ecdf_test.to_csv(dir+"ecdf/"+obs+"_"+ensemble+"-sub_mon-p095_"+str(start)+"-"+str(end)+"_ecdf_test.csv")

cmip
gpcc
gpcp
mswep
mesaclip
gpcc
gpcp
mswep
spear
gpcc
gpcp
mswep


### Comparisons over time:

In [10]:
## loop through different time periods for mon-p095
for start in np.arange(1930, 1979, 1):  
    print(start)
    ## compare gpcc to models (cmip, mesaclip, spear)
    end = start+41
    gpcc_trend = _utils.read_trends(dir, "gpcc", "mon-p095", start, end)
    cmip_trend = _utils.read_trends(dir, "cmip", "mon-p095", start, end).sel(sim = model_var_dict["cmip_mon_onevar"])
    ecdf_result = ecdf_xr(cmip_trend, gpcc_trend)
    ecdf_result.to_netcdf(dir+"ecdf/gpcc_cmip-sub_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

    mesaclip_trend = _utils.read_trends(dir, "mesaclip", "mon-p095", start, end)
    ecdf_result = ecdf_xr(mesaclip_trend, gpcc_trend)
    ecdf_result.to_netcdf(dir+"ecdf/gpcc_mesaclip_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

    spear_trend = _utils.read_trends(dir, "spear", "mon-p095", start, end)
    ecdf_result = ecdf_xr(spear_trend, gpcc_trend)
    ecdf_result.to_netcdf(dir+"ecdf/gpcc_spear_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

    ### with 2020 as end date
    end = 2020
    gpcc_trend = _utils.read_trends(dir, "gpcc", "mon-p095", start, end)
    cmip_trend = _utils.read_trends(dir, "cmip", "mon-p095", start, end).sel(sim = model_var_dict["cmip_mon_onevar"])
    ecdf_result = ecdf_xr(cmip_trend, gpcc_trend)
    ecdf_result.to_netcdf(dir+"ecdf/gpcc_cmip-sub_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

    mesaclip_trend = _utils.read_trends(dir, "mesaclip", "mon-p095", start, end)
    ecdf_result = ecdf_xr(mesaclip_trend, gpcc_trend)
    ecdf_result.to_netcdf(dir+"ecdf/gpcc_mesaclip_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

    spear_trend = _utils.read_trends(dir, "spear", "mon-p095", start, end)
    ecdf_result = ecdf_xr(spear_trend, gpcc_trend)
    ecdf_result.to_netcdf(dir+"ecdf/gpcc_spear_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

1930
1931


FileNotFoundError: [Errno 2] No such file or directory: '/Users/fvdav22/Documents/research/extreme-precip-trends/mnt_processed_data/gpcc_trends/gpcc_mon-p095_1931-1972_trend.nc'

In [8]:
## loop through different time periods for rx1day
for start in np.arange(1950, 1975, 1): 
    print(start)
    ## compare regen to models (cmip, mesaclip, spear)
    for end in np.arange(start+30, 2016):
        regen_trend = _utils.read_trends(dir, "regen", "rx1day", start, end)
        cmip_trend = _utils.read_trends(dir, "cmip", "rx1day", start, end).sel(sim = model_var_dict["cmip_day_onevar"])
        ecdf_result = ecdf_xr(cmip_trend, regen_trend)
        ecdf_result.to_netcdf(dir+"ecdf/regen_cmip-sub_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")
    
        mesaclip_trend = _utils.read_trends(dir, "mesaclip", "rx1day", start, end)
        ecdf_result = ecdf_xr(mesaclip_trend, regen_trend)
        ecdf_result.to_netcdf(dir+"ecdf/regen_mesaclip_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")
    
        spear_trend = _utils.read_trends(dir, "spear", "rx1day", start, end)
        ecdf_result = ecdf_xr(spear_trend, regen_trend)
        ecdf_result.to_netcdf(dir+"ecdf/regen_spear_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")

    ### with 2016 as end date
    #end = 2016
    #regen_trend = _utils.read_trends(dir, "regen", "rx1day", start, end)
    #cmip_trend = _utils.read_trends(dir, "cmip", "rx1day", start, end).sel(sim = model_var_dict["cmip_day_onevar"])
    #ecdf_result = ecdf_xr(cmip_trend, regen_trend)
    #ecdf_result.to_netcdf(dir+"ecdf/regen_cmip-sub_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")

    #mesaclip_trend = _utils.read_trends(dir, "mesaclip", "rx1day", start, end)
    #ecdf_result = ecdf_xr(mesaclip_trend, regen_trend)
    #ecdf_result.to_netcdf(dir+"ecdf/regen_mesaclip_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")

    #spear_trend = _utils.read_trends(dir, "spear", "rx1day", start, end)
    #ecdf_result = ecdf_xr(spear_trend, regen_trend)
    #ecdf_result.to_netcdf(dir+"ecdf/regen_spear_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")
    

1950


ValueError: must supply at least one object to concatenate

In [13]:
dir

'../mnt_processed_data/'

In [14]:
x = _utils.read_trends(dir, "cmip", "rx1day", start, end)

In [11]:
end

2001

### Subsampling

In [ ]:
## Rx1day
start = 1979
end = 2020

for ensemble in ["cmip", "spear"]:
    print(ensemble)
    model_trend = mask(_utils.read_trends(dir, ensemble, "rx1day", start, end))
    
    for obs in ["cpc", "mswep"]:
        print(obs)
        obs_trend = mask(_utils.read_trends(dir, obs, "rx1day", start, end))

        if ensemble == "cmip":
            model_trend = model_trend.sel(sim = model_var_dict["cmip_day_onevar"])
            ecdf_result = test_ecdf_subsample(model_trend, obs_trend, N = 30)
            ecdf_result.to_csv(dir+"ecdf/"+obs+"_"+ensemble+"_rx1day_"+str(start)+"-"+str(end)+"_ecdf_subsample30.csv")
            
        ecdf_result = test_ecdf_subsample(model_trend, obs_trend, N = 9)
        ecdf_result.to_csv(dir+"ecdf/"+obs+"_"+ensemble+"_rx1day_"+str(start)+"-"+str(end)+"_ecdf_subsample9.csv")

In [ ]:
## monthly p95
start = 1979
end = 2020

for ensemble in ["cmip", "spear"]:
    print(ensemble)
    model_trend = mask(_utils.read_trends(dir, ensemble, "mon-p095", start, end))
    
    for obs in ["gpcc", "gpcp", "mswep"]:
        print(obs)
        obs_trend = mask(_utils.read_trends(dir, obs, "mon-p095", start, end))

        if ensemble == "cmip":
            model_trend = model_trend.sel(sim = model_var_dict["cmip_mon_onevar"])
            ecdf_result = test_ecdf_subsample(model_trend, obs_trend, N = 30)
            ecdf_result.to_csv(dir+"ecdf/"+obs+"_"+ensemble+"_mon-p095_"+str(start)+"-"+str(end)+"_ecdf_subsample30.csv")
            
        ecdf_result = test_ecdf_subsample(model_trend, obs_trend, N = 9)
        ecdf_result.to_csv(dir+"ecdf/"+obs+"_"+ensemble+"_mon-p095_"+str(start)+"-"+str(end)+"_ecdf_subsample9.csv")